# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and analyzing the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library. All dataset components such as record sets, fields, and columns are referenced by their Croissant `@id` identifiers, ensuring reproducibility and clarity.

### Dataset Source
The dataset is loaded from the Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset and metadata using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")
print(f"Published: {metadata.datePublished}")

## 2. Data Overview
List all available record sets, fields, and columns, using their `@id` fields. This helps to understand the dataset structure and the identifiers to use for data extraction.

In [ ]:
# Overview of record sets and their fields (@id)
from pprint import pprint

rs = metadata.recordSets
print(f"Found {len(rs)} record set(s) in the dataset.")
record_set_ids = [record_set["@id"] for record_set in rs]
for i, record_set in enumerate(rs):
    print(f"\nRecord Set {i + 1}: {record_set['name']} (@id={record_set['@id']})")
    print("  Description:", record_set.get('description', 'N/A'))
    # List fields in this record set
    if "fields" in record_set:
        print("  Fields:")
        for field in record_set["fields"]:
            print(f"    - {field['name']} (@id={field['@id']}, dataType={field.get('dataType', 'N/A')})")
    # List columns (for tabular data)
    if "columns" in record_set:
        print("  Columns:")
        for column in record_set["columns"]:
            print(f"    - {column['name']} (@id={column['@id']})")

## 3. Data Extraction
Now, load the data for each record set by its `@id`. Access the records and place them in pandas DataFrames for easy manipulation and analysis.

In [ ]:
# Extract data from each record set into a DataFrame
dataframes = {}

for record_set in metadata.recordSets:
    rs_id = record_set["@id"]
    print(f"Loading records for record set '{record_set['name']}' (@id={rs_id}) ...")
    records = list(dataset.records(record_set=rs_id))
    if records:  # Only add if nonempty
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"  -> DataFrame shape: {df.shape}, columns: {df.columns.tolist()}")
    else:
        print("  -> No records found in this record set.")

# For demonstration, pick the first available record set with data
if dataframes:
    main_rs_id = list(dataframes.keys())[0]
    print(f"\nExample record set (@id={main_rs_id}):")
    print(dataframes[main_rs_id].head())
else:
    print("No data record sets found.")

## 4. Exploratory Data Analysis (EDA)
Let's process the largest/most relevant available record set. We will filter, normalize, and group by selected fields, referencing their `@id` throughout.

In [ ]:
# Identify a numeric field for analysis by checking dtypes or by inspecting fields with dataType='Float' or 'Integer'
import numpy as np

if dataframes:
    df = dataframes[main_rs_id].copy()
    print(f"Columns in main record set (@id={main_rs_id}): {df.columns.tolist()}")

    # Try to select a numeric field. If columns are known: set by '@id' or column name, otherwise guess first float or int column
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_cols:
        # Try to convert columns with numbers
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
            except:
                continue
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_cols:
        print("No numeric columns found for EDA.")
    else:
        numeric_field = numeric_cols[0]
        print(f"Using numeric field '@id' or column: {numeric_field}")

        # Filtering
        threshold = df[numeric_field].quantile(0.75)  # As an example threshold
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records where {numeric_field} > {threshold:.2f} (top 25%): {filtered_df.shape[0]} records")
        
        # Normalizing
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        
        # Try grouping by a field -- prefer a non-numeric field, otherwise show unique values
        nonnum_cols = [c for c in df.columns if c not in numeric_cols]
        if nonnum_cols:
            group_field = nonnum_cols[0]
            print(f"Grouping by field: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False).to_frame()
            print(grouped_df.head())
        else:
            print("No non-numeric grouping field found.")
else:
    print("No data available for EDA.")

## 5. Visualization
Let's visualize the distribution of the chosen numeric field and its relationship to a grouping field if possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_cols:
    # Distribution
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field], bins=30, kde=True)
    plt.title(f"Distribution of '{numeric_field}' in record set '@id={main_rs_id}'")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()
    # Boxplot by group field
    if nonnum_cols:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=df[nonnum_cols[0]], y=df[numeric_field])
        plt.title(f"Boxplot of {numeric_field} grouped by {nonnum_cols[0]}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we loaded the *Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells* dataset using its Croissant schema, explored its structure via `@id` references, extracted data for each record set, and performed initial data analysis and visualization.

**Key findings:**
- The dataset contains mass spectrometry protein quantification and annotation fields.
- Data exploration was performed by referencing `@id` identifiers throughout the workflow, ensuring reproducibility and clarity as defined by the Croissant metadata standard.
- Filtering and normalization of a numeric field, and basic visualizations, can be readily accomplished. More advanced analyses can be conducted by following the Croissant structure for reliable data integration and processing.

For further research, consult the [dataset source](https://sen.science/doi/10.71728/senscience.vq4a-28xa) and original documentation.